In [5]:
import json
import os
from datetime import datetime
from typing import List, Dict, Optional


MEMBERS_FILE = "members.json"
RECORDS_FILE = "records.json"


# ==================== 成员管理 ====================

def load_members() -> List[str]:
    """加载成员列表"""
    if not os.path.exists(MEMBERS_FILE):
        return []
    with open(MEMBERS_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def save_members(members: List[str]) -> None:
    """保存成员列表"""
    with open(MEMBERS_FILE, "w", encoding="utf-8") as f:
        json.dump(members, f, ensure_ascii=False, indent=2)


def view_members() -> None:
    """查看所有成员"""
    members = load_members()
    if not members:
        print(" 当前没有成员")
    else:
        print(f" 当前成员（共{len(members)}人）：")
        for i, name in enumerate(members, 1):
            print(f"  {i}. {name}")


def add_member(name: str) -> None:
    """添加成员"""
    if not name.strip():
        print(" 姓名不能为空")
        return
    members = load_members()
    if name in members:
        print(f" {name} 已存在")
        return
    members.append(name)
    save_members(members)
    print(f"✅ 已添加：{name}")


def remove_member(name: str) -> None:
    """删除成员"""
    members = load_members()
    if name not in members:
        print(f" 找不到 {name}")
        return
    members.remove(name)
    save_members(members)
    print(f" 已删除：{name}")


def update_member(old_name: str, new_name: str) -> None:
    """修改成员姓名"""
    if not new_name.strip():
        print(" 新姓名不能为空")
        return
    members = load_members()
    if old_name not in members:
        print(f" 找不到 {old_name}")
        return
    if new_name in members and new_name != old_name:
        print(f" {new_name} 已存在")
        return
    idx = members.index(old_name)
    members[idx] = new_name
    save_members(members)
    print(f" 已修改：{old_name} → {new_name}")


# ==================== 费用记录管理 ====================

def load_records() -> List[Dict]:
    """加载历史记录"""
    if not os.path.exists(RECORDS_FILE):
        return []
    with open(RECORDS_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def save_records(records: List[Dict]) -> None:
    """保存历史记录"""
    with open(RECORDS_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def get_last_reading() -> Optional[Dict]:
    """获取上次的读数"""
    records = load_records()
    if not records:
        return None
    return records[-1]


def add_record(water_reading: float, elec_reading: float, 
               water_rate: float = 3.5, elec_rate: float = 0.6) -> None:
    """
    录入费用（通过读数计算）
    water_reading: 水表读数（吨）
    elec_reading: 电表读数（度）
    water_rate: 水费单价（元/吨）
    elec_rate: 电费单价（元/度）
    """
    month = datetime.now().strftime("%Y年%m月")
    
    # 检查当月是否已有记录
    records = load_records()
    for r in records:
        if r["month"] == month:
            print(f" {month} 已有记录，请勿重复录入")
            return
    
    # 获取上次读数
    last = get_last_reading()
    
    if last:
        water_used = water_reading - last["water_reading"]
        elec_used = elec_reading - last["elec_reading"]
        water_cost = round(water_used * water_rate, 2)
        elec_cost = round(elec_used * elec_rate, 2)
    else:
        # 首次录入，没有上次读数，直接按总量计算
        water_cost = round(water_reading * water_rate, 2)
        elec_cost = round(elec_reading * elec_rate, 2)
    
    # 获取当前成员列表快照
    members = load_members()
    
    record = {
        "month": month,
        "water_reading": water_reading,
        "elec_reading": elec_reading,
        "water_cost": water_cost,
        "elec_cost": elec_cost,
        "total_cost": water_cost + elec_cost,
        "members": members.copy(),
        "timestamp": datetime.now().isoformat()
    }
    
    records.append(record)
    save_records(records)
    
    print(f" 已录入：{month}")
    print(f"    水表读数：{water_reading}吨 → 水费：{water_cost}元")
    print(f"    电表读数：{elec_reading}度 → 电费：{elec_cost}元")
    print(f"    合计：{water_cost + elec_cost}元")
    

def calculate_split(method: str = "average", custom_weights: Dict[str, float] = None) -> None:
    """
    计算分摊费用
    method: "average" 平均分摊, "ratio" 按人数比例分摊
    custom_weights: 自定义权重，如 {"张三": 1.5, "李四": 1.0, "王五": 1.0}
    """
    records = load_records()
    if not records:
        print(" 请先录入费用")
        return
    
    latest = records[-1]
    members = latest.get("members", load_members())
    
    if not members:
        print(" 请先添加成员")
        return
    
    total_cost = latest["total_cost"]
    
    print(f"\n {latest['month']} 分摊结果（分摊方式：{method}）：")
    print(f"    成员：{', '.join(members)}")
    print(f"    总费用：{total_cost}元（水费{latest['water_cost']}元 + 电费{latest['elec_cost']}元）")
    
    if method == "average":
        # 平均分摊
        per_person = round(total_cost / len(members), 2)
        print(f"    每人分摊：{per_person}元")
        print("\n   明细：")
        for name in members:
            print(f"     - {name}：{per_person}元")
            
    elif method == "ratio":
        # 按人数比例分摊（根据各房间/家庭人数）
        if custom_weights is None:
            # 默认每个人权重为1
            weights = {name: 1.0 for name in members}
        else:
            weights = custom_weights
        
        # 确保所有成员都有权重
        for name in members:
            if name not in weights:
                weights[name] = 1.0
        
        total_weight = sum(weights[name] for name in members)
        print(f"\n    权重比例：")
        for name in members:
            print(f"     - {name}：{weights[name]}")
        
        print("\n   分摊明细：")
        for name in members:
            share = round(total_cost * weights[name] / total_weight, 2)
            print(f"     - {name}：{share}元")
    
    else:
        print(f" 不支持的分摊方式：{method}")


# ==================== 历史查询 ====================

def view_history() -> None:
    """查看历史记录"""
    records = load_records()
    if not records:
        print(" 暂无历史记录")
        return
    
    print("\n ===== 历史记录 =====\n")
    for i, r in enumerate(reversed(records), 1):
        print(f"  {i}. {r['month']}")
        print(f"      水费：{r['water_cost']}元（读数：{r['water_reading']}吨）")
        print(f"      电费：{r['elec_cost']}元（读数：{r['elec_reading']}度）")
        print(f"      合计：{r['total_cost']}元")
        print(f"      成员：{', '.join(r.get('members', []))}")
        print()


def view_record_detail(index: int) -> None:
    """查看某条记录的详细信息"""
    records = load_records()
    if not records or index < 1 or index > len(records):
        print(" 记录不存在")
        return
    
    r = records[index - 1]
    print(f"\n ===== {r['month']} 详细记录 =====\n")
    print(f"  水表读数：{r['water_reading']}吨 → 水费：{r['water_cost']}元")
    print(f"  电表读数：{r['elec_reading']}度 → 电费：{r['elec_cost']}元")
    print(f"  总费用：{r['total_cost']}元")
    print(f"  成员：{', '.join(r.get('members', []))}")
    print(f"  录入时间：{r.get('timestamp', '未知')}")


# ==================== 交互菜单 ====================

def print_menu():
    """打印主菜单"""
    print("\n" + "=" * 50)
    print("    宿舍水电费分摊计算系统")
    print("=" * 50)
    print("  1. 成员管理")
    print("  2. 录入费用读数")
    print("  3. 计算分摊")
    print("  4. 查看历史记录")
    print("  5. 查看记录详情")
    print("  6. 设置分摊方式")
    print("  0. 退出")
    print("=" * 50)


def member_management():
    """成员管理子菜单"""
    while True:
        print("\n📋 ===== 成员管理 =====\n")
        print("  1. 查看所有成员")
        print("  2. 添加成员")
        print("  3. 修改成员")
        print("  4. 删除成员")
        print("  0. 返回主菜单")
        
        choice = input("\n请选择：").strip()
        
        if choice == "0":
            break
        elif choice == "1":
            view_members()
        elif choice == "2":
            name = input("请输入成员姓名：").strip()
            add_member(name)
        elif choice == "3":
            old_name = input("请输入要修改的姓名：").strip()
            new_name = input("请输入新姓名：").strip()
            update_member(old_name, new_name)
        elif choice == "4":
            view_members()
            name = input("请输入要删除的姓名：").strip()
            remove_member(name)
        else:
            print(" 无效选项")


def input_record():
    """录入费用读数"""
    print("\n ===== 录入费用读数 =====\n")
    
    try:
        water_reading = float(input("请输入水表当前读数（吨）：").strip())
        elec_reading = float(input("请输入电表当前读数（度）：").strip())
        
        # 可选：设置单价
        use_default = input("使用默认单价吗？（水3.5元/吨，电0.6元/度）(y/n)：").strip().lower()
        if use_default == "n":
            water_rate = float(input("请输入水费单价（元/吨）：").strip())
            elec_rate = float(input("请输入电费单价（元/度）：").strip())
        else:
            water_rate = 3.5
            elec_rate = 0.6
        
        add_record(water_reading, elec_reading, water_rate, elec_rate)
        
    except ValueError:
        print(" 请输入有效数字")


def split_menu():
    """分摊计算菜单"""
    print("\n ===== 分摊计算 =====\n")
    print("  1. 平均分摊")
    print("  2. 按人数比例分摊")
    
    choice = input("\n请选择分摊方式：").strip()
    
    if choice == "1":
        calculate_split("average")
    elif choice == "2":
        # 获取权重
        members = load_members()
        if not members:
            print(" 请先添加成员")
            return
        
        print("\n输入各成员的权重（默认为1，按回车跳过）：")
        weights = {}
        for name in members:
            w = input(f"  {name} 的权重：").strip()
            if w:
                try:
                    weights[name] = float(w)
                except ValueError:
                    print(f"  无效输入，{name} 使用默认权重1")
                    weights[name] = 1.0
            else:
                weights[name] = 1.0
        
        calculate_split("ratio", weights)
    else:
        print(" 无效选项")


def main():
    """主程序入口"""
    print("\n 欢迎使用宿舍水电费分摊计算系统")
    print("   （数据保存在 members.json 和 records.json）")
    
    current_method = "average"
    
    while True:
        print_menu()
        choice = input("\n请选择（0-6）：").strip()
        
        if choice == "0":
            print("\n 再见！")
            break
        elif choice == "1":
            member_management()
        elif choice == "2":
            input_record()
        elif choice == "3":
            split_menu()
        elif choice == "4":
            view_history()
        elif choice == "5":
            records = load_records()
            if records:
                view_history()
                try:
                    idx = int(input("\n请输入要查看的记录编号：").strip())
                    view_record_detail(idx)
                except ValueError:
                    print(" 请输入有效数字")
            else:
                print(" 暂无记录")
        elif choice == "6":
            print(f"\n当前分摊方式：{current_method}")
            print("  1. 平均分摊")
            print("  2. 按人数比例分摊")
            c = input("请选择：").strip()
            if c == "1":
                current_method = "average"
                print(" 已切换为平均分摊")
            elif c == "2":
                current_method = "ratio"
                print(" 已切换为按人数比例分摊")
            else:
                print(" 无效选项")
        else:
            print(" 无效选项")


In [6]:
# ==================== 快速测试（演示用） ====================

def demo():
    """快速演示功能"""
    print("=" * 50)
    print("   快速演示模式")
    print("=" * 50)
    
    # 清理旧数据
    if os.path.exists(MEMBERS_FILE):
        os.remove(MEMBERS_FILE)
    if os.path.exists(RECORDS_FILE):
        os.remove(RECORDS_FILE)
    
    # 1. 添加成员
    print("\n 添加成员...")
    add_member("张三")
    add_member("李四")
    add_member("王五")
    
    # 2. 查看成员
    print("\n 查看成员...")
    view_members()
    
    # 3. 录入费用（模拟：月初读数）
    print("\n 录入费用（月初读数）...")
    add_record(100, 200)
    
    # 4. 录入下月费用（通过差值计算）
    print("\n 录入下月费用（月末读数）...")
    add_record(115, 320)  # 用水15吨，用电120度
    
    # 5. 平均分摊
    print("\n 平均分摊...")
    calculate_split("average")
    
    # 6. 按比例分摊
    print("\n 按人数比例分摊...")
    calculate_split("ratio", {"张三": 1.5, "李四": 1.0, "王五": 1.0})
    
    # 7. 查看历史
    print("\n 查看历史...")
    view_history()
    
    # 8. 修改成员
    print("\n 修改成员（李四→赵六）...")
    update_member("李四", "赵六")
    view_members()


if __name__ == "__main__":
    # 取消注释下一行可以进入演示模式
    demo()
    
    # 正常启动交互式菜单
    #main()

   快速演示模式

 添加成员...
✅ 已添加：张三
✅ 已添加：李四
✅ 已添加：王五

 查看成员...
 当前成员（共3人）：
  1. 张三
  2. 李四
  3. 王五

 录入费用（月初读数）...
 已录入：2026年07月
    水表读数：100吨 → 水费：350.0元
    电表读数：200度 → 电费：120.0元
    合计：470.0元

 录入下月费用（月末读数）...
 2026年07月 已有记录，请勿重复录入

 平均分摊...

 2026年07月 分摊结果（分摊方式：average）：
    成员：张三, 李四, 王五
    总费用：470.0元（水费350.0元 + 电费120.0元）
    每人分摊：156.67元

   明细：
     - 张三：156.67元
     - 李四：156.67元
     - 王五：156.67元

 按人数比例分摊...

 2026年07月 分摊结果（分摊方式：ratio）：
    成员：张三, 李四, 王五
    总费用：470.0元（水费350.0元 + 电费120.0元）

    权重比例：
     - 张三：1.5
     - 李四：1.0
     - 王五：1.0

   分摊明细：
     - 张三：201.43元
     - 李四：134.29元
     - 王五：134.29元

 查看历史...

 ===== 历史记录 =====

  1. 2026年07月
      水费：350.0元（读数：100吨）
      电费：120.0元（读数：200度）
      合计：470.0元
      成员：张三, 李四, 王五


 修改成员（李四→赵六）...
 已修改：李四 → 赵六
 当前成员（共3人）：
  1. 张三
  2. 赵六
  3. 王五
